# Launch Morphology Mutation

Desktop multi-SWC morphology editor with drag-box selection.

Core controls in the window:
- `r` drag-box: add to selection
- `t`/`y`: thin/thicken
- `g`: grow branch
- `d`: click-to-draw branch to a picked 3D point
- `b`: split selected edges
- `a`: reparent using last two selected
- `x`: detach selected
- `j`: add pre->post connection spec
- `s`: save mutation bundle (SWCs + manifest + validation + connection specs)
- `z`: undo


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys


def _find_work_root() -> Path:
    candidates = []
    env_root = os.environ.get('DIGIFLY_VIP_GLIA_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root))
    try:
        candidates.append(Path.cwd())
    except Exception:
        pass
    try:
        cwd = Path.cwd()
        for base in [cwd] + list(cwd.parents):
            candidates.append(base / 'Phase 2' / 'apps' / 'VIP_Glia_Sim')
            candidates.append(base / 'apps' / 'VIP_Glia_Sim')
    except Exception:
        pass
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue
        for root in [candidate] + list(candidate.parents):
            if (root / 'tools' / 'morphology_mutation_app.py').exists():
                return root
    raise RuntimeError('Could not locate VIP_Glia_Sim app root. Set DIGIFLY_VIP_GLIA_ROOT.')


WORK_ROOT = _find_work_root()
PHASE2_ROOT = Path(os.environ.get('DIGIFLY_PHASE2_ROOT', str(WORK_ROOT.parents[1]))).expanduser().resolve()
APP_PATH = WORK_ROOT / 'tools' / 'morphology_mutation_app.py'
SWC_DIR = Path(os.environ.get('DIGIFLY_SWC_DIR', str(PHASE2_ROOT / 'data' / 'export_swc'))).expanduser().resolve()

# Prefer conda python for desktop/VTK dependencies; fallback to current kernel python.
_py_cands = [
    '/opt/anaconda3/bin/python3.12',
    '/opt/anaconda3/bin/python3',
    '/opt/anaconda3/bin/python',
    sys.executable,
]
PYTHON_BIN = None
for _pc in _py_cands:
    _pp = Path(str(_pc)).expanduser()
    if _pp.exists():
        PYTHON_BIN = _pp.resolve()
        break
if PYTHON_BIN is None:
    PYTHON_BIN = Path(sys.executable).resolve()

NEURON_IDS = [10000, 10002, 10068, 10110, 11446, 11654, ]
MUTATION_TAG = 'split'
OUTPUT_ROOT = WORK_ROOT / 'notebooks' / 'debug' / 'outputs'

# Mutation defaults
THIN_FACTOR = 0.8
THICK_FACTOR = 1.2
GROW_LENGTH_UM = 18.0
GROW_SEGMENTS = 4
GROW_RADIUS_SCALE = 0.85

# Inter-neuron connection-spec defaults
CONNECTION_CHEMICAL = 1
CONNECTION_GAP = 0
CONNECTION_GAP_MODE = 'none'  # none|non_rectifying|rectifying

# View defaults for the desktop app
RENDER_MODE = 'neuroglancer'          # 'tube', 'skeleton', or 'neuroglancer' (key 'w' toggles standard modes; key '1' toggles neuroglancer)
NEUROGLANCER_QUALITY = 'ultra'        # 'auto', 'balanced', 'high', or 'ultra' (max clarity; heavier render)
START_SOLO = True                     # recommended for neuroglancer volume mode; press '-' in the app to toggle all neurons
START_NEURON_ID = int(NEURON_IDS[0]) if NEURON_IDS else None
SKELETON_LINE_WIDTH = 6.0     # used when render_mode='skeleton' and by the slider
VISUAL_STYLE = 'classic'      # legacy flag kept for CLI compatibility; app uses classic only

# Optional flow-movie export inputs
FLOW_RUN_DIR = None           # explicit override; leave as None to auto-pick a matching finished run
FLOW_RUNS_ROOT = WORK_ROOT / 'notebooks' / 'debug' / 'runs'
FLOW_FOCUS_PAIR = (10000, 10068)
FLOW_FPS = 20
FLOW_FRAME_STRIDE = 4
FLOW_SPEED_UM_PER_MS = 220.0
FLOW_PULSE_SIGMA_MS = 0.6
FLOW_SYN_DELAY_MS = None
FLOW_THRESHOLD_MV = 0.0
FLOW_MAX_MS = 25.0


def _records_neuron_ids(run_dir: Path) -> set[int]:
    rec_path = run_dir / 'records.csv'
    if not rec_path.exists():
        return set()
    header = rec_path.open('r', encoding='utf-8').readline().strip().split(',')
    out = set()
    for col in header:
        col = str(col).strip()
        if col.endswith('_soma_v'):
            nid = col[:-7]
            if nid.isdigit():
                out.add(int(nid))
    return out


def _auto_pick_flow_run_dir(runs_root: Path, neuron_ids: list[int]) -> Path | None:
    runs_root = Path(runs_root).expanduser().resolve()
    if not runs_root.exists():
        return None

    wanted = {int(x) for x in neuron_ids}
    candidates = []
    for run_dir in sorted([p for p in runs_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True):
        cfg_path = run_dir / 'config.json'
        rec_path = run_dir / 'records.csv'
        if not cfg_path.exists() or not rec_path.exists():
            continue
        try:
            rec_ids = _records_neuron_ids(run_dir)
        except Exception:
            continue
        overlap = len(wanted & rec_ids)
        if overlap == 0:
            continue
        full_match = wanted.issubset(rec_ids)
        name = run_dir.name.lower()
        baseline_pref = int(('baseline' in name) and ('reduced' not in name) and ('coalesced' not in name))
        candidates.append((int(full_match), baseline_pref, overlap, run_dir.stat().st_mtime, run_dir))

    if not candidates:
        return None

    candidates.sort(reverse=True)
    return candidates[0][-1]


if FLOW_RUN_DIR is None:
    FLOW_RUN_DIR = _auto_pick_flow_run_dir(FLOW_RUNS_ROOT, NEURON_IDS)
else:
    FLOW_RUN_DIR = Path(FLOW_RUN_DIR).expanduser().resolve()

print('PYTHON_BIN =', PYTHON_BIN)
print('APP_PATH    =', APP_PATH)
print('SWC_DIR     =', SWC_DIR)
print('NEURON_IDS  =', NEURON_IDS)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RENDER_MODE =', RENDER_MODE)
print('NEUROGLANCER_QUALITY =', NEUROGLANCER_QUALITY)
print('START_SOLO =', START_SOLO, 'START_NEURON_ID =', START_NEURON_ID)
print('SKELETON_LINE_WIDTH =', SKELETON_LINE_WIDTH)
print('VISUAL_STYLE =', VISUAL_STYLE)
print('FLOW_RUN_DIR =', FLOW_RUN_DIR)
print('FLOW_FOCUS_PAIR =', FLOW_FOCUS_PAIR)


In [ ]:
cmd = [
    str(PYTHON_BIN),
    str(APP_PATH),
    '--swc-dir', str(SWC_DIR),
    '--phase2-root', str(PHASE2_ROOT),
    '--neuron-ids', ','.join(str(int(x)) for x in NEURON_IDS),
    '--output-root', str(OUTPUT_ROOT),
    '--tag', str(MUTATION_TAG),
    '--thin-factor', str(THIN_FACTOR),
    '--thick-factor', str(THICK_FACTOR),
    '--grow-length-um', str(GROW_LENGTH_UM),
    '--grow-segments', str(GROW_SEGMENTS),
    '--grow-radius-scale', str(GROW_RADIUS_SCALE),
    '--connection-chemical', str(CONNECTION_CHEMICAL),
    '--connection-gap', str(CONNECTION_GAP),
    '--connection-gap-mode', str(CONNECTION_GAP_MODE),
    '--render-mode', str(RENDER_MODE),
    '--skeleton-line-width', str(SKELETON_LINE_WIDTH),
    '--visual-style', str(VISUAL_STYLE),
    '--neuroglancer-quality', str(NEUROGLANCER_QUALITY),
]
if START_SOLO:
    cmd.append('--start-solo')
if START_NEURON_ID is not None:
    cmd.extend(['--start-neuron-id', str(int(START_NEURON_ID))])
if FLOW_RUN_DIR is not None:
    cmd.extend(['--flow-run-dir', str(Path(FLOW_RUN_DIR).expanduser().resolve())])
if FLOW_FOCUS_PAIR is not None:
    cmd.extend(['--flow-focus-pair', ','.join(str(int(x)) for x in FLOW_FOCUS_PAIR)])
cmd.extend(['--flow-fps', str(FLOW_FPS)])
cmd.extend(['--flow-frame-stride', str(FLOW_FRAME_STRIDE)])
cmd.extend(['--flow-speed-um-per-ms', str(FLOW_SPEED_UM_PER_MS)])
cmd.extend(['--flow-pulse-sigma-ms', str(FLOW_PULSE_SIGMA_MS)])
if FLOW_SYN_DELAY_MS is not None:
    cmd.extend(['--flow-syn-delay-ms', str(FLOW_SYN_DELAY_MS)])
cmd.extend(['--flow-threshold-mv', str(FLOW_THRESHOLD_MV)])
cmd.extend(['--flow-max-ms', str(FLOW_MAX_MS)])
print(shlex.join(cmd))


In [ ]:
import time

env = os.environ.copy()
LOG_DIR = OUTPUT_ROOT / '_launcher_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LAUNCH_LOG = LOG_DIR / f"morphology_mutation_launcher_{int(time.time())}.log"

with open(LAUNCH_LOG, 'w', encoding='utf-8') as _lf:
    mutation_proc = subprocess.Popen(
        cmd,
        cwd=str(WORK_ROOT),
        env=env,
        stdout=_lf,
        stderr=subprocess.STDOUT,
    )

time.sleep(1.0)
rc = mutation_proc.poll()
if rc is None:
    print(f'Launched morphology mutation app (PID={mutation_proc.pid}).')
    print('Launcher log =', LAUNCH_LOG)
else:
    print(f'Launch failed early (exit={rc}). Log: {LAUNCH_LOG}')
    try:
        _txt = LAUNCH_LOG.read_text(encoding='utf-8')
        print('--- launcher log tail ---')
        print(_txt[-4000:])
    except Exception as _e:
        print('Could not read launcher log:', _e)


In [ ]:
# Optional: stop a still-running desktop process.
if 'mutation_proc' in globals() and mutation_proc.poll() is None:
    mutation_proc.terminate()
    print('Sent terminate to PID', mutation_proc.pid)
else:
    print('No running mutation process found in this kernel.')


## Find Latest Mutation Manifest And Wire Into Sim Notebooks

Run this after saving (`s`) inside the mutation app.


In [ ]:
from pathlib import Path
import json

cand_dirs = sorted(
    [p for p in OUTPUT_ROOT.glob('morphology_mutation_*') if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

LATEST_MUTATION_MANIFEST = None
for d in cand_dirs:
    m = d / 'morphology_mutation_manifest.json'
    if m.exists():
        LATEST_MUTATION_MANIFEST = m.resolve()
        break

if LATEST_MUTATION_MANIFEST is None:
    raise FileNotFoundError(f'No morphology mutation manifest found under: {OUTPUT_ROOT}')

print('LATEST_MUTATION_MANIFEST =', LATEST_MUTATION_MANIFEST)

m = json.loads(LATEST_MUTATION_MANIFEST.read_text(encoding="utf-8"))
print('neuron_ids =', m.get('neuron_ids'))
print('phase2_overlay_dir =', m.get('phase2_overlay_dir'))
print('n_ops =', len(m.get('operations') or []))
print('n_connections =', len(m.get('connections') or []))

print("\n# Paste into glia_simulation.ipynb controls:")
print("MUTATION_USE_MANIFEST = True")
print(f'MUTATION_MANIFEST_JSON = r"{LATEST_MUTATION_MANIFEST}"')
print("MUTATION_INCLUDE_CONNECTIONS = True")

print("\n# Paste into glia_force_ko_sweep_parallel.ipynb controls:")
print("SWEEP_MUTATION_USE_MANIFEST = True")
print(f'SWEEP_MUTATION_MANIFEST_JSON = r"{LATEST_MUTATION_MANIFEST}"')
print("SWEEP_MUTATION_INCLUDE_CONNECTIONS = True")
